# AI Architecture Guardian v5.1: GNN + pre-merge history (GPU)

**Before running:** Settings, then Accelerator **GPU T4 x2**, then **Internet ON**. Add the `ai-guardian-v51` dataset with *Add Input*. Then **Save Version, then Save & Run All**.

Trains three regularisation settings (5 seeds each) with the history features. The one used in the report is chosen by **validation** PR-AUC only. The diff embeddings come from the v5 run, so they are not recomputed here. Output: `/kaggle/working/v51_results.zip`.

In [ ]:
!pip install -q torch_geometric
import torch, torch_geometric
print('torch', torch.__version__, '| PyG', torch_geometric.__version__, '| GPU:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Turn on the GPU accelerator in Settings'

In [ ]:
import gzip, pathlib, shutil, zipfile
INPUT = pathlib.Path('/kaggle/input')
def find(*names):
    for name in names:
        hits = sorted(INPUT.rglob(name))
        if hits:
            return hits[0]
    return None
WORK = pathlib.Path('/kaggle/working/ai_guardian'); OUT = pathlib.Path('/kaggle/working/out')
OUT.mkdir(parents=True, exist_ok=True)
if WORK.exists(): shutil.rmtree(WORK)
# Kaggle sometimes unpacks uploaded archives itself, so accept both forms.
code_zip = find('code.zip')
if code_zip is not None and code_zip.is_file():
    zipfile.ZipFile(code_zip).extractall(WORK)
else:
    src_dir = next(p for p in INPUT.rglob('src') if (p / 'ml' / 'coupling_data.py').is_file())
    shutil.copytree(src_dir.parent, WORK)
packed = find('graphs_v5.pt.gz', 'graphs_v5.pt')
assert packed is not None, 'graphs_v5.pt.gz not found: did you add the dataset with Add Input?'
if packed.suffix == '.gz':
    with gzip.open(packed, 'rb') as src, open('/kaggle/working/graphs_v5.pt', 'wb') as dst:
        shutil.copyfileobj(src, dst)
else:
    shutil.copy(packed, '/kaggle/working/graphs_v5.pt')
DIFFS = find('diffs_v5.jsonl'); SPLIT = find('coupling_split_v5.json')
assert DIFFS and SPLIT, 'diffs_v5.jsonl or coupling_split_v5.json missing from the dataset'
shutil.copy(SPLIT, OUT / 'coupling_split_v5.json')
print(sorted(p.name for p in WORK.iterdir()), DIFFS, SPLIT)

In [ ]:
%cd /kaggle/working/ai_guardian
CONFIGS = {
    'gnn_v51':       '--hidden-channels 64 --dropout 0.3 --weight-decay 3e-3',
    'gnn_v51_small': '--hidden-channels 32 --dropout 0.5 --weight-decay 1e-2',
    'gnn_v51_wide':  '--hidden-channels 128 --dropout 0.4 --weight-decay 5e-3',
}
for name, options in CONFIGS.items():
    print('=====', name, options, flush=True)
    !python -W ignore scripts/train_coupling_gnn.py --graphs /kaggle/working/graphs_v5.pt --split {SPLIT} --output-dir {OUT}/{name} --seeds 1 2 3 4 5 {options}

In [ ]:
import json, statistics
rows = []
for name in CONFIGS:
    summary = json.load(open(OUT / name / 'metrics.json'))
    assert summary['settings']['history_dim'] > 0, f'{name} trained WITHOUT history: is history.json in the packed data?'
    validation = statistics.mean(r['validation']['pr_auc_average_precision'] for r in summary['runs'])
    rows.append((validation, name))
    print(f"{name:15s} validation PR-AUC {validation:.4f}")
best = max(rows)[1]
print('Chosen by validation:', best)
(OUT / 'chosen_by_validation.txt').write_text(best + '\n')
shutil.make_archive('/kaggle/working/v51_results', 'zip', OUT)
print('Download /kaggle/working/v51_results.zip')
# Keep only the results zip as Kaggle output: saving the 1 GB unpacked graphs and the code
# folder makes "Save Version" spend 20+ minutes uploading files nobody needs.
shutil.rmtree(WORK, ignore_errors=True)
shutil.rmtree(OUT, ignore_errors=True)
pathlib.Path('/kaggle/working/graphs_v5.pt').unlink(missing_ok=True)
print('Output folder now:', sorted(p.name for p in pathlib.Path('/kaggle/working').iterdir()))